# SHV-R1A-R5A0 — Actual Custom-Autograd MLP Saved-State Integration Ruler

R4R3R4R1 formally qualified the MLP saved-state representation:

\[
\boxed{
Z=LN_2(U)+FP16\ rstd_2
}
\]

under fresh Support, Promotion and sealed Guard.

The qualified heterogeneous physical schedule is frozen:

\[
\begin{array}{c|c}
Layer & Saved\ Z\ codec\\ \hline
0&z6g128\\
1&z6g32\\
2&z6g64\\
3&z6g32\\
4&z6g32\\
5&z7g128\\
6&z7g128\\
7&z6g16\\
8&z6g16\\
9&z6g16\\
10&z6g32\\
11&z6g32
\end{array}
\]

R5A0 stops doing representation search.

It asks a systems question:

\[
\boxed{
\text{Can the qualified state replace real PyTorch autograd residency and reduce actual CUDA memory?}
}
\]

## Important implementation decision

A generic `saved_tensors_hooks(pack_hook, unpack_hook)` is not sufficient by itself for this MLP state.

Native LayerNorm backward expects its own saved input/statistics, while SHV has qualified a different sufficient state:

\[
(Z,rstd_2)
\]

Therefore this round uses a real:

\[
\boxed{\texttt{torch.autograd.Function}}
\]

whose forward is native/exact but whose backward stores only packed \(Z\)+\(rstd_2\), recomputes MLP, and executes the qualified explicit LayerNorm backward.

`saved_tensors_hooks` is still used as an audit instrument to verify what the custom Function actually saves.

This round integrates **MLP/LN2 only**. The qualified Attention parent is not yet replaced in the live graph.


In [ ]:
import importlib.util, subprocess, sys
need=[]
for pkg in ["transformers","datasets"]:
    if importlib.util.find_spec(pkg) is None:
        need.append(pkg)
if need:
    subprocess.check_call([sys.executable,"-m","pip","install","-q","transformers","datasets"])
print("dependencies ready")


In [ ]:
import os, gc, math, json, random, platform, hashlib, re, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import transformers, datasets
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

assert torch.cuda.is_available(), "CUDA GPU required."
DEVICE=torch.device("cuda")

SEED=20260909
MODEL_ID="gpt2"
SEQ_LEN=128
HIDDEN=768
NHEADS=12
HEAD_DIM=64

SUPPORT_N=12
PROMOTION_N=8
GUARD_N=8

# Fresh after the completed R4R3R4R1 panels.
LANG_SUPPORT_SCAN_START=336
LANG_PROMOTION_SCAN_START=184
LANG_GUARD_SCAN_START=200
CODE_TRAIN_SCAN_START=428

# Integration fidelity probes. Representation selection is already frozen.
INTEGRATION_SUPPORT_SAMPLES_PER_DOMAIN=2
INTEGRATION_PROMOTION_SAMPLES_PER_DOMAIN=2
INTEGRATION_GUARD_SAMPLES_PER_DOMAIN=2

SUPPORT_GRAD_MIN=0.997
PROMO_GRAD_MIN=0.995
GUARD_GRAD_MIN=0.995

FORWARD_IDENTITY_NRMSE_MAX=1e-6

# Actual-memory qualification for this unfused Python/PyTorch prototype.
FORWARD_RESIDENT_REDUCTION_MIN=0.10
FULL_STEP_PEAK_REDUCTION_MIN=0.05

MEMORY_BENCH_BATCH=8
MEMORY_BENCH_REPEATS=3

FROZEN_Z_MODES={
    0:"z6g128",
    1:"z6g32",
    2:"z6g64",
    3:"z6g32",
    4:"z6g32",
    5:"z7g128",
    6:"z7g128",
    7:"z6g16",
    8:"z6g16",
    9:"z6g16",
    10:"z6g32",
    11:"z6g32",
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

OUT=Path("/content/SHV_R1A_R5A0_RESULTS") if Path("/content").exists() else Path("./SHV_R1A_R5A0_RESULTS")
OUT.mkdir(parents=True,exist_ok=True)

prop=torch.cuda.get_device_properties(0)
ENV={
    "python":sys.version,
    "platform":platform.platform(),
    "torch":torch.__version__,
    "torch_cuda":torch.version.cuda,
    "transformers":transformers.__version__,
    "datasets":datasets.__version__,
    "gpu":torch.cuda.get_device_name(0),
    "gpu_total_gib":prop.total_memory/1024**3,
}
(OUT/"environment.json").write_text(json.dumps(ENV,indent=2),encoding="utf-8")
ENV


## Pre-registration

### Frozen representation

No layer codec may change in R5A0.

### Forward contract

Compression happens only after native \(Z\) has already been consumed by forward.

The custom autograd Function must reproduce native:

\[
U+MLP(LN_2(U))
\]

with:

\[
NRMSE\le10^{-6}
\]

before backward.

### Gradient contract

Compare native autograd and SHV custom autograd for:

- block input \(U\);
- `ln_2.weight`;
- `ln_2.bias`;
- `c_fc.weight`;
- `c_fc.bias`;
- `c_proj.weight`;
- `c_proj.bias`.

Fresh Support requires every cosine:

\[
\ge0.997
\]

Promotion / Guard:

\[
\ge0.995
\]

### Saved-state contract

A `saved_tensors_hooks` census around the custom Function must show that no full-size FP16/FP32 \(Z\) tensor is retained by the Function.

The stored activation payload must consist of:

- packed uint8 Z code;
- FP16 group scales;
- FP16 rstd.

### Actual CUDA-memory contract

A 12-layer accumulated MLP/LN2 backward-residency benchmark uses real post-Attention \(U\) tensors from 8 fresh sequences.

Compare native versus SHV:

1. CUDA allocated bytes resident immediately after forward;
2. full forward+backward peak allocated bytes.

R5A0 requires at least:

\[
10\%
\]

forward-resident reduction and:

\[
5\%
\]

full-step peak reduction.

These are intentionally modest integration gates.

Throughput is recorded but is **not** a formal gate because the current packer/backward is an unfused Python/PyTorch prototype.

### Scope

Passing R5A0 qualifies the **MLP/LN2 live-autograd integration**.

It does not yet claim whole-model SHV because Attention live-autograd replacement remains for R5A1/R5B.


In [ ]:
PREREG={
    "experiment":"SHV-R1A-R5A0",
    "parent":"R4R3R4R1 formal PASS",
    "scope":"actual custom-autograd integration of MLP/LN2 saved state",
    "frozen_z_modes":FROZEN_Z_MODES,
    "forward_identity_nrmse_max":FORWARD_IDENTITY_NRMSE_MAX,
    "support_gradient_cos_min":SUPPORT_GRAD_MIN,
    "promotion_gradient_cos_min":PROMO_GRAD_MIN,
    "guard_gradient_cos_min":GUARD_GRAD_MIN,
    "memory":{
        "benchmark_batch":MEMORY_BENCH_BATCH,
        "forward_resident_reduction_min":FORWARD_RESIDENT_REDUCTION_MIN,
        "full_step_peak_reduction_min":FULL_STEP_PEAK_REDUCTION_MIN,
        "throughput_is_formal_gate":False,
    },
    "fresh_scan_starts":{
        "language_support_train":LANG_SUPPORT_SCAN_START,
        "language_promotion_validation":LANG_PROMOTION_SCAN_START,
        "language_guard_test":LANG_GUARD_SCAN_START,
        "code_mbpp_train":CODE_TRAIN_SCAN_START,
    },
    "next_if_pass":"live Attention integration, then fused Decode-on-Compute",
}
(OUT/"preregistration.json").write_text(json.dumps(PREREG,indent=2),encoding="utf-8")
PREREG


In [ ]:
def cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

def nrmse(a,b,eps=1e-12):
    return float((a.float()-b.float()).pow(2).mean().sqrt()/
                 (a.float().pow(2).mean().sqrt()+eps))

def cosine_flat(a,b,eps=1e-12):
    a=a.detach().float().reshape(-1)
    b=b.detach().float().reshape(-1)
    return float(torch.dot(a,b)/(a.norm()*b.norm()+eps))

@torch.no_grad()
def qint8_rows(x):
    xf=x.float()
    s=(xf.abs().amax(dim=1,keepdim=True)/127.0).clamp_min(1e-8)
    q=torch.round(xf/s).clamp(-127,127).to(torch.int8)
    return q,s.to(torch.float16)

@torch.no_grad()
def dqint8_rows(q,s,dtype=torch.float16):
    return q.to(dtype)*s.to(dtype)

def row_int8_bytes(rows,cols):
    return rows*cols + rows*2


## Fresh real-data panels and strict hash audit


In [ ]:
WIKITEXT_REV="00aa25585682d4957f9e86edc73f59be7419af99"
wiki=load_dataset("Salesforce/wikitext","wikitext-2-raw-v1",revision=WIKITEXT_REV)
mbpp=load_dataset("arjunguha/mbpp")
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,dtype=torch.float16).to(DEVICE)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

EOS=tokenizer.eos_token_id
NLAYERS=len(model.transformer.h)
NSTATES=NLAYERS+1
assert model.config.n_embd==HIDDEN
assert model.config.n_head==NHEADS
assert HIDDEN//NHEADS==HEAD_DIM

gamma_min=min(float(model.transformer.h[l].ln_1.weight.detach().abs().min()) for l in range(NLAYERS))
assert gamma_min>1e-6


def iter_texts(domain,split):
    if domain=="language":
        for row in wiki[split]:
            t=(row.get("text") or "").strip()
            if t:
                yield t
    else:
        for row in mbpp[split]:
            parts=[(row.get("text") or "").strip(),(row.get("code") or "").strip()]
            tests=row.get("test_list")
            if tests:
                parts.append("\n".join(map(str,tests)))
            t="\n".join(x for x in parts if x)
            if t:
                yield t


def iter_exact_blocks(domain,split):
    buf=[]
    produced=0
    for text in iter_texts(domain,split):
        ids=tokenizer(text,add_special_tokens=False)["input_ids"]
        if not ids:
            continue
        buf.extend(ids+[EOS])
        while len(buf)>=SEQ_LEN:
            block=torch.tensor(buf[:SEQ_LEN],dtype=torch.long)
            del buf[:SEQ_LEN]
            yield produced,block
            produced+=1


def pack_exact(domain,split,nblocks,skip_blocks=0):
    out=[]
    for idx,block in iter_exact_blocks(domain,split):
        if idx<skip_blocks:
            continue
        out.append(block)
        if len(out)>=nblocks:
            return out
    raise RuntimeError({"insufficient_blocks":{
        "domain":domain,"split":split,"skip_blocks":skip_blocks,
        "requested":nblocks,"selected":len(out)
    }})


def bhash(x):
    return hashlib.sha256(x.numpy().tobytes()).hexdigest()


# Historical scientific-exposure set.
legacy=[
    ("train",0,16),("validation",0,8),("test",0,8),
    ("train",24,16),("validation",16,8),("test",32,8),
    ("train",40,16),("validation",24,8),("test",40,8),
    ("train",56,12),("validation",32,8),("test",48,8),
    ("train",72,12),("validation",40,8),("test",56,8),
    ("train",88,12),("validation",48,8),("test",64,8),
    ("train",104,12),("validation",56,8),("test",72,8),
    ("train",120,12),("validation",64,8),("test",80,8),
    ("train",136,12),("validation",72,8),("test",88,8),
    ("train",152,12),("validation",80,8),("test",96,8),
    ("train",168,12),("validation",88,8),("test",104,8),
    ("train",184,12),("validation",96,8),("test",112,8),
    ("train",200,12),("validation",104,8),("test",120,8),
    ("train",216,12),("validation",112,8),("test",128,8),
    ("train",232,12),("validation",120,8),("test",136,8),
]
prior_specs=[]
for split,skip,n in legacy:
    for domain in ["language","code"]:
        prior_specs.append((domain,split,skip,n))

# First incomplete R4R3 run reached data construction / later Support work.
prior_specs += [
    # First incomplete R4R3 run.
    ("language","train",248,12),("code","train",248,12),
    ("language","validation",128,8),("code","validation",128,5),
    ("language","test",144,8),("code","test",144,8),

    # Completed R4R3R1R1 allocator run: all selected panels are
    # conservatively treated as exposed.
    ("language","train",264,12),
    ("language","validation",136,8),
    ("language","test",152,8),
    ("code","train",260,28),

    # Completed R4R3R2 allocator panels; all are conservatively exposed.
    ("language","train",276,12),
    ("language","validation",144,8),
    ("language","test",160,8),
    ("code","train",288,28),

    # Completed R4R3R3 panels.
    ("language","train",288,12),
    ("language","validation",152,8),
    ("language","test",168,8),
    ("code","train",316,28),

    # Completed R4R3R3R1 panels.
    ("language","train",300,12),
    ("language","validation",160,8),
    ("language","test",176,8),
    ("code","train",344,28),

    # Completed R4R3R4 allocator panels. Promotion/Guard were not
    # model-evaluated, but their hashes were allocated and archived,
    # so all panels are conservatively treated as exposed.
    ("language","train",312,12),
    ("language","validation",168,8),
    ("language","test",184,8),
    ("code","train",372,28),

    # Completed R4R3R4R1 panels.
    ("language","train",324,12),
    ("language","validation",176,8),
    ("language","test",192,8),
    ("code","train",400,28),
]

PRIOR_HASHES=set()
for domain,split,skip,n in prior_specs:
    PRIOR_HASHES.update(bhash(x) for x in pack_exact(domain,split,n,skip))


CURRENT_RESERVED_HASHES=set()
FRESHNESS_AUDIT=[]


def allocate_fresh_blocks(phase,domain,split,scan_start,nblocks):
    selected=[]
    selected_indices=[]
    selected_hashes=[]
    prior_duplicate_indices=[]
    current_duplicate_indices=[]
    scanned_after_start=0

    for idx,block in iter_exact_blocks(domain,split):
        if idx<scan_start:
            continue
        scanned_after_start+=1
        h=bhash(block)
        if h in PRIOR_HASHES:
            prior_duplicate_indices.append(int(idx))
            continue
        if h in CURRENT_RESERVED_HASHES:
            current_duplicate_indices.append(int(idx))
            continue
        selected.append(block)
        selected_indices.append(int(idx))
        selected_hashes.append(h)
        CURRENT_RESERVED_HASHES.add(h)
        if len(selected)>=nblocks:
            break

    if len(selected)<nblocks:
        raise RuntimeError({"fresh_allocator_insufficient_unique_blocks":{
            "phase":phase,"domain":domain,"split":split,
            "scan_start":scan_start,"requested":nblocks,
            "selected":len(selected),"scanned_after_start":scanned_after_start,
            "prior_duplicate_count":len(prior_duplicate_indices),
            "current_duplicate_count":len(current_duplicate_indices),
        }})

    record={
        "phase":phase,
        "domain":domain,
        "split":split,
        "scan_start":int(scan_start),
        "requested":int(nblocks),
        "selected_source_block_indices":selected_indices,
        "selected_hashes":selected_hashes,
        "prior_duplicate_count":len(prior_duplicate_indices),
        "prior_duplicate_source_indices":prior_duplicate_indices,
        "current_duplicate_count":len(current_duplicate_indices),
        "current_duplicate_source_indices":current_duplicate_indices,
        "last_selected_source_index":int(selected_indices[-1]),
    }
    FRESHNESS_AUDIT.append(record)
    return selected


PANELS={"support":{},"promotion":{},"guard":{}}

# Language: preserve official split roles.
PANELS["support"]["language"]=allocate_fresh_blocks(
    "support","language","train",LANG_SUPPORT_SCAN_START,SUPPORT_N
)
PANELS["promotion"]["language"]=allocate_fresh_blocks(
    "promotion","language","validation",LANG_PROMOTION_SCAN_START,PROMOTION_N
)
PANELS["guard"]["language"]=allocate_fresh_blocks(
    "guard","language","test",LANG_GUARD_SCAN_START,GUARD_N
)

# Code: deterministic single MBPP-train stream, globally reserved.
PANELS["support"]["code"]=allocate_fresh_blocks(
    "support","code","train",CODE_TRAIN_SCAN_START,SUPPORT_N
)
PANELS["promotion"]["code"]=allocate_fresh_blocks(
    "promotion","code","train",CODE_TRAIN_SCAN_START,PROMOTION_N
)
PANELS["guard"]["code"]=allocate_fresh_blocks(
    "guard","code","train",CODE_TRAIN_SCAN_START,GUARD_N
)


new_sets={
    phase:set(bhash(x) for domain in ["language","code"] for x in PANELS[phase][domain])
    for phase in PANELS
}
all_new=new_sets["support"]|new_sets["promotion"]|new_sets["guard"]
prior_overlap=all_new&PRIOR_HASHES
support_promotion_overlap=new_sets["support"]&new_sets["promotion"]
support_guard_overlap=new_sets["support"]&new_sets["guard"]
promotion_guard_overlap=new_sets["promotion"]&new_sets["guard"]

FINAL_HASH_AUDIT={
    "prior_overlap_count":len(prior_overlap),
    "support_promotion_overlap_count":len(support_promotion_overlap),
    "support_guard_overlap_count":len(support_guard_overlap),
    "promotion_guard_overlap_count":len(promotion_guard_overlap),
    "all_passed":bool(
        len(prior_overlap)==0 and
        len(support_promotion_overlap)==0 and
        len(support_guard_overlap)==0 and
        len(promotion_guard_overlap)==0
    ),
}
if not FINAL_HASH_AUDIT["all_passed"]:
    raise RuntimeError({"final_fresh_hash_audit_failed":FINAL_HASH_AUDIT})

(OUT/"freshness_allocator_audit.json").write_text(
    json.dumps({"allocations":FRESHNESS_AUDIT,"final_hash_audit":FINAL_HASH_AUDIT},indent=2),
    encoding="utf-8"
)

DATA_CONTRACT={
    "real_sources_loaded":True,
    "allocator_rule":"first N globally unseen hashes in deterministic source order",
    "allocator_uses_only_hash_freshness":True,
    "allocator_uses_model_outputs":False,
    "allocator_uses_task_content":False,
    "exact_length_no_padding":True,
    "no_overlap_with_all_fully_specified_prior_exposed_blocks":True,
    "new_support_promotion_guard_disjoint":True,
    "language_protocol":"official train/validation/test streams with hash-only duplicate rejection",
    "code_protocol":"fresh MBPP-train block holdout with hash-only duplicate rejection",
    "code_official_split_generalization_claimed":False,
    "minimum_abs_ln1_gamma":gamma_min,
    "freshness_allocator_audit_file":"freshness_allocator_audit.json",
}
(OUT/"data_contract.json").write_text(json.dumps(DATA_CONTRACT,indent=2),encoding="utf-8")

print("Fresh allocator PASS")
for row in FRESHNESS_AUDIT:
    print({
        "phase":row["phase"],
        "domain":row["domain"],
        "split":row["split"],
        "scan_start":row["scan_start"],
        "selected_indices":row["selected_source_block_indices"],
        "prior_duplicates_skipped":row["prior_duplicate_count"],
        "current_duplicates_skipped":row["current_duplicate_count"],
    })
DATA_CONTRACT


## Capture native GPT-2 residual-stream states


In [ ]:
@torch.no_grad()
def capture_states(ids_cpu):
    states={}
    handles=[]

    for li,block in enumerate(model.transformer.h):
        def make_pre(idx):
            def hook(mod,inp):
                states[idx]=inp[0].detach().squeeze(0).contiguous()
            return hook
        handles.append(block.register_forward_pre_hook(make_pre(li)))

    def last_hook(mod,inp,out):
        h=out[0] if isinstance(out,(tuple,list)) else out
        states[NLAYERS]=h.detach().squeeze(0).contiguous()

    handles.append(model.transformer.h[-1].register_forward_hook(last_hook))

    try:
        _=model(input_ids=ids_cpu.unsqueeze(0).to(DEVICE),use_cache=False)
    finally:
        for h in handles:
            h.remove()

    assert sorted(states)==list(range(NSTATES))
    return [states[i] for i in range(NSTATES)]

@torch.no_grad()
def collect_cpu(split):
    out={"language":[],"code":[]}

    for domain in ["language","code"]:
        for i,ids in enumerate(PANELS[split][domain]):
            st=capture_states(ids)
            out[domain].append([x.cpu() for x in st])
            del st

            if (i+1)%4==0:
                print(split,domain,i+1)

        cleanup()

    return out

SUPPORT_CPU=collect_cpu("support")
PROMOTION_CPU=collect_cpu("promotion")
GUARD_CPU=collect_cpu("guard")


## Specialized physical 6/7-bit saved-Z packer

R4R3 used a general variable-width ruler.

R5A0 uses a lower-workspace fixed-bit implementation:

- 6-bit: 4 values -> 3 bytes;
- 7-bit: 8 values -> 7 bytes.

The packed byte stream itself is saved by autograd.


In [ ]:
def parse_z_mode(mode):
    m=re.fullmatch(r"z([67])g(\d+)",mode)
    if m is None:
        raise ValueError(mode)
    return int(m.group(1)),int(m.group(2))


def fixed_pack_layout(bits):
    if bits==6:
        return 4,3
    if bits==7:
        return 8,7
    raise ValueError(bits)


@torch.no_grad()
def pack_fixed_uint(u,bits):
    u=u.reshape(-1)
    vals_per_word,bytes_per_word=fixed_pack_layout(bits)

    assert u.numel()%vals_per_word==0, {
        "numel":u.numel(),
        "bits":bits,
        "vals_per_word":vals_per_word,
    }

    groups=u.to(torch.int64).reshape(-1,vals_per_word)

    word=torch.zeros(
        groups.shape[0],
        dtype=torch.int64,
        device=groups.device,
    )

    for i in range(vals_per_word):
        word |= (
            groups[:,i] <<
            (bits*i)
        )

    out=torch.empty(
        (groups.shape[0],bytes_per_word),
        dtype=torch.uint8,
        device=groups.device,
    )

    for j in range(bytes_per_word):
        out[:,j]=(
            (word>>(8*j))&
            255
        ).to(torch.uint8)

    return out.reshape(-1).contiguous()


@torch.no_grad()
def unpack_fixed_uint(packed,bits,numel):
    vals_per_word,bytes_per_word=fixed_pack_layout(bits)

    assert numel%vals_per_word==0

    p=packed.reshape(-1,bytes_per_word).to(torch.int64)

    word=torch.zeros(
        p.shape[0],
        dtype=torch.int64,
        device=p.device,
    )

    for j in range(bytes_per_word):
        word |= (
            p[:,j] <<
            (8*j)
        )

    mask=(1<<bits)-1

    vals=[]

    for i in range(vals_per_word):
        vals.append(
            (
                word>>
                (bits*i)
            )&
            mask
        )

    return (
        torch.stack(
            vals,
            dim=1,
        )
        .reshape(-1)[:numel]
    )


@torch.no_grad()
def pack_saved_z(Z,mode):
    bits,group=parse_z_mode(mode)

    shape=tuple(Z.shape)
    rows=int(np.prod(shape[:-1]))
    cols=shape[-1]

    assert cols==HIDDEN
    assert cols%group==0

    qmax=(1<<(bits-1))-1

    xf=Z.float().reshape(
        rows,
        cols//group,
        group,
    )

    scale=(
        xf.abs()
        .amax(
            dim=-1,
            keepdim=True,
        )/
        float(qmax)
    ).clamp_min(1e-8)

    q=torch.round(
        xf/scale
    ).clamp(
        -qmax,
        qmax,
    ).to(torch.int16)

    u=(
        q.to(torch.int64)+
        qmax
    ).reshape(-1)

    packed=pack_fixed_uint(
        u,
        bits,
    )

    stored_scale=scale.squeeze(-1).to(torch.float16).contiguous()

    expected_payload=(
        rows*cols*bits
    )//8

    assert packed.numel()==expected_payload, {
        "mode":mode,
        "actual":packed.numel(),
        "expected":expected_payload,
    }

    return packed,stored_scale


@torch.no_grad()
def unpack_saved_z(packed,scale,mode,shape):
    bits,group=parse_z_mode(mode)
    rows=int(np.prod(shape[:-1]))
    cols=shape[-1]
    qmax=(1<<(bits-1))-1

    numel=rows*cols

    u=unpack_fixed_uint(
        packed,
        bits,
        numel,
    )

    q=(
        u.to(torch.int64)-
        qmax
    ).reshape(
        rows,
        cols//group,
        group,
    )

    Zhat=(
        q.float()*
        scale.float().unsqueeze(-1)
    ).reshape(shape)

    return Zhat


# Exact integer round-trip audit.
PACK_AUDIT={}

with torch.no_grad():
    gen=torch.Generator(device=DEVICE)
    gen.manual_seed(SEED+99173)

    for bits in [6,7]:
        qmax=(1<<(bits-1))-1
        vals_per_word,_=fixed_pack_layout(bits)
        n=65536
        n=(n//vals_per_word)*vals_per_word

        q=torch.randint(
            -qmax,
            qmax+1,
            (n,),
            generator=gen,
            device=DEVICE,
            dtype=torch.int64,
        )

        u=q+qmax
        p=pack_fixed_uint(u,bits)
        u2=unpack_fixed_uint(p,bits,n)

        PACK_AUDIT[str(bits)]={
            "roundtrip_exact":bool(torch.equal(u,u2)),
            "packed_bytes":int(p.numel()),
            "expected_bytes":int(n*bits//8),
        }

        if not PACK_AUDIT[str(bits)]["roundtrip_exact"]:
            raise RuntimeError({"fixed_bit_packer_failed":PACK_AUDIT})

        if p.numel()!=n*bits//8:
            raise RuntimeError({"fixed_bit_byte_count_failed":PACK_AUDIT})

(OUT/"fixed_bit_packer_audit.json").write_text(
    json.dumps(PACK_AUDIT,indent=2),
    encoding="utf-8"
)

PACK_AUDIT


## Actual custom autograd Function

Forward uses the real GPT-2 `ln_2` and MLP modules.

Only after the exact forward output has been formed does SHV pack \(Z\).

Backward decodes \(\hat Z\), recomputes the MLP under autograd, and applies the explicit LayerNorm backward formula.


In [ ]:
class SHVLN2MLPFunction(torch.autograd.Function):

    @staticmethod
    def forward(
        ctx,
        U,
        ln_w,
        ln_b,
        cfc_w,
        cfc_b,
        cproj_w,
        cproj_b,
        layer_idx,
        mode,
    ):
        layer_idx=int(layer_idx)
        block=model.transformer.h[layer_idx]

        # Parameter identity contract.
        assert ln_w.data_ptr()==block.ln_2.weight.data_ptr()
        assert ln_b.data_ptr()==block.ln_2.bias.data_ptr()
        assert cfc_w.data_ptr()==block.mlp.c_fc.weight.data_ptr()
        assert cfc_b.data_ptr()==block.mlp.c_fc.bias.data_ptr()
        assert cproj_w.data_ptr()==block.mlp.c_proj.weight.data_ptr()
        assert cproj_b.data_ptr()==block.mlp.c_proj.bias.data_ptr()

        with torch.no_grad():
            Z=block.ln_2(U)
            M=block.mlp(Z)
            out=U+M

            packed,scale=pack_saved_z(
                Z,
                mode,
            )

            var=U.float().var(
                dim=-1,
                unbiased=False,
                keepdim=True,
            )

            rstd=torch.rsqrt(
                var+
                float(block.ln_2.eps)
            ).to(torch.float16)

        ctx.save_for_backward(
            packed,
            scale,
            rstd,
        )

        # Parameters already live permanently in the model; these are references,
        # not new activation copies.
        ctx.param_refs=(
            ln_w,
            ln_b,
            cfc_w,
            cfc_b,
            cproj_w,
            cproj_b,
        )

        ctx.layer_idx=layer_idx
        ctx.mode=mode
        ctx.z_shape=tuple(Z.shape)
        ctx.u_dtype=U.dtype

        return out


    @staticmethod
    def backward(
        ctx,
        grad_output,
    ):
        packed,scale,rstd=ctx.saved_tensors

        (
            ln_w,
            ln_b,
            cfc_w,
            cfc_b,
            cproj_w,
            cproj_b,
        )=ctx.param_refs

        block=model.transformer.h[
            ctx.layer_idx
        ]

        Zhat=unpack_saved_z(
            packed,
            scale,
            ctx.mode,
            ctx.z_shape,
        )

        G=grad_output.float()

        # Recompute MLP and its parameter/input gradients.
        with torch.enable_grad():
            Zreq=(
                Zhat.detach()
                .to(
                    block.ln_2.weight.dtype
                )
                .requires_grad_(True)
            )

            Mhat=block.mlp(
                Zreq
            )

            surrogate=(
                Mhat.float()*
                G
            ).sum()

            (
                dZ,
                gcfc_w,
                gcfc_b,
                gcproj_w,
                gcproj_b,
            )=torch.autograd.grad(
                surrogate,
                [
                    Zreq,
                    cfc_w,
                    cfc_b,
                    cproj_w,
                    cproj_b,
                ],
                retain_graph=False,
                create_graph=False,
                allow_unused=False,
            )

        gamma=ln_w.detach().float()
        beta=ln_b.detach().float()

        gamma_min=float(
            gamma.abs().min()
        )

        if gamma_min<=1e-6:
            raise RuntimeError({
                "ln2_gamma_too_small":{
                    "layer":ctx.layer_idx,
                    "min_abs_gamma":gamma_min,
                }
            })

        xhat=(
            Zhat.float()-
            beta
        )/gamma

        dy=dZ.float()

        gln_w=(
            dy*
            xhat
        ).sum(
            dim=tuple(
                range(
                    dy.ndim-1
                )
            )
        )

        gln_b=dy.sum(
            dim=tuple(
                range(
                    dy.ndim-1
                )
            )
        )

        dxhat=dy*gamma

        mean_dxhat=dxhat.mean(
            dim=-1,
            keepdim=True,
        )

        mean_dxhat_xhat=(
            dxhat*
            xhat
        ).mean(
            dim=-1,
            keepdim=True,
        )

        dU_ln=(
            rstd.float()*
            (
                dxhat-
                mean_dxhat-
                xhat*
                mean_dxhat_xhat
            )
        )

        dU=(
            G+
            dU_ln
        ).to(
            ctx.u_dtype
        )

        return (
            dU,
            gln_w.to(ln_w.dtype),
            gln_b.to(ln_b.dtype),
            gcfc_w.to(cfc_w.dtype),
            gcfc_b.to(cfc_b.dtype),
            gcproj_w.to(cproj_w.dtype),
            gcproj_b.to(cproj_b.dtype),
            None,
            None,
        )


def layer_params(layer):
    block=model.transformer.h[layer]
    return [
        block.ln_2.weight,
        block.ln_2.bias,
        block.mlp.c_fc.weight,
        block.mlp.c_fc.bias,
        block.mlp.c_proj.weight,
        block.mlp.c_proj.bias,
    ]


def with_param_grad_flags(params,enabled=True):
    old=[bool(p.requires_grad) for p in params]
    for p in params:
        p.requires_grad_(enabled)
    return old


def restore_param_grad_flags(params,flags):
    for p,f in zip(params,flags):
        p.requires_grad_(f)


@torch.no_grad()
def native_post_attention_u(st_cpu,layer):
    H=st_cpu[layer].to(DEVICE)
    block=model.transformer.h[layer]

    N=block.ln_1(
        H.to(
            block.ln_1.weight.dtype
        )
    )

    A=block.attn(
        N.unsqueeze(0),
        use_cache=False,
    )[0].squeeze(0)

    return (
        H+
        A.to(H.dtype)
    ).contiguous()


def deterministic_integration_upstream(layer,domain,phase):
    gen=torch.Generator(device=DEVICE)
    gen.manual_seed(
        SEED+
        29009*layer+
        181*sum(ord(c) for c in domain)+
        223*sum(ord(c) for c in phase)
    )

    g=torch.randn(
        (SEQ_LEN,HIDDEN),
        generator=gen,
        device=DEVICE,
        dtype=torch.float32,
    )

    return g/(g.norm()+1e-12)


def native_case(U0,layer,upstream):
    block=model.transformer.h[layer]
    params=layer_params(layer)
    old=with_param_grad_flags(params,True)

    try:
        with torch.enable_grad():
            U=(
                U0.detach()
                .clone()
                .to(block.ln_2.weight.dtype)
                .requires_grad_(True)
            )

            out=(
                U+
                block.mlp(
                    block.ln_2(U)
                )
            )

            loss=(
                out.float()*
                upstream
            ).sum()

            grads=torch.autograd.grad(
                loss,
                [U]+params,
                retain_graph=False,
                create_graph=False,
                allow_unused=False,
            )

        return (
            out.detach().float(),
            [
                g.detach().float()
                for g in grads
            ],
        )

    finally:
        restore_param_grad_flags(params,old)


def shv_case(U0,layer,mode,upstream):
    params=layer_params(layer)
    old=with_param_grad_flags(params,True)

    try:
        with torch.enable_grad():
            U=(
                U0.detach()
                .clone()
                .to(
                    model.transformer.h[layer].ln_2.weight.dtype
                )
                .requires_grad_(True)
            )

            out=SHVLN2MLPFunction.apply(
                U,
                *params,
                layer,
                mode,
            )

            loss=(
                out.float()*
                upstream
            ).sum()

            grads=torch.autograd.grad(
                loss,
                [U]+params,
                retain_graph=False,
                create_graph=False,
                allow_unused=False,
            )

        return (
            out.detach().float(),
            [
                g.detach().float()
                for g in grads
            ],
        )

    finally:
        restore_param_grad_flags(params,old)


GRAD_NAMES=[
    "block_input",
    "ln2_weight",
    "ln2_bias",
    "cfc_weight",
    "cfc_bias",
    "cproj_weight",
    "cproj_bias",
]


## Fresh autograd fidelity qualification

This does not re-select codecs.

It verifies that the *actual custom autograd implementation* reproduces the already-qualified analytical backward behavior on fresh data.


In [ ]:
def run_phase_fidelity(
    phase,
    panel_cpu,
    samples_per_domain,
    grad_min,
):
    rows=[]

    for layer in range(12):
        mode=FROZEN_Z_MODES[layer]

        print(
            phase,
            "layer",
            layer,
            mode,
        )

        for domain in [
            "language",
            "code",
        ]:
            for sample in range(
                min(
                    samples_per_domain,
                    len(panel_cpu[domain]),
                )
            ):
                U0=native_post_attention_u(
                    panel_cpu[domain][sample],
                    layer,
                )

                upstream=deterministic_integration_upstream(
                    layer,
                    domain,
                    phase,
                )

                native_out,native_grads=native_case(
                    U0,
                    layer,
                    upstream,
                )

                shv_out,shv_grads=shv_case(
                    U0,
                    layer,
                    mode,
                    upstream,
                )

                row={
                    "phase":phase,
                    "layer":layer,
                    "domain":domain,
                    "sample":sample,
                    "z_codec":mode,
                    "forward_nrmse":nrmse(
                        native_out,
                        shv_out,
                    ),
                }

                for name,g,h in zip(
                    GRAD_NAMES,
                    native_grads,
                    shv_grads,
                ):
                    row[
                        name+"_grad_cos"
                    ]=cosine_flat(
                        g,
                        h,
                    )

                    row[
                        name+"_grad_relerr"
                    ]=float(
                        (
                            h-g
                        ).norm()/
                        (
                            g.norm()+
                            1e-12
                        )
                    )

                rows.append(
                    row
                )

                del U0,native_out,shv_out,native_grads,shv_grads
                cleanup()

    df=pd.DataFrame(rows)

    max_forward=float(
        df.forward_nrmse.max()
    )

    grad_cols=[
        name+"_grad_cos"
        for name in GRAD_NAMES
    ]

    min_grad=float(
        min(
            df[col].min()
            for col in grad_cols
        )
    )

    passed=bool(
        max_forward<=FORWARD_IDENTITY_NRMSE_MAX
        and
        min_grad>=grad_min
    )

    return df,{
        "phase":phase,
        "rows":len(df),
        "max_forward_nrmse":max_forward,
        "min_gradient_cosine":min_grad,
        "required_gradient_cosine":grad_min,
        "passed":passed,
    }


SUPPORT_FID,SUPPORT_FID_SUM=run_phase_fidelity(
    "support_integration",
    SUPPORT_CPU,
    INTEGRATION_SUPPORT_SAMPLES_PER_DOMAIN,
    SUPPORT_GRAD_MIN,
)

SUPPORT_FID.to_csv(
    OUT/"support_custom_autograd_fidelity.csv",
    index=False,
)

print(
    SUPPORT_FID_SUM
)


if SUPPORT_FID_SUM["passed"]:
    PROMO_FID,PROMO_FID_SUM=run_phase_fidelity(
        "promotion_integration",
        PROMOTION_CPU,
        INTEGRATION_PROMOTION_SAMPLES_PER_DOMAIN,
        PROMO_GRAD_MIN,
    )
else:
    PROMO_FID=pd.DataFrame()
    PROMO_FID_SUM={
        "phase":"promotion_integration",
        "passed":False,
        "sealed":True,
    }

PROMO_FID.to_csv(
    OUT/"promotion_custom_autograd_fidelity.csv",
    index=False,
)

print(
    PROMO_FID_SUM
)


if PROMO_FID_SUM.get("passed",False):
    GUARD_FID,GUARD_FID_SUM=run_phase_fidelity(
        "guard_integration",
        GUARD_CPU,
        INTEGRATION_GUARD_SAMPLES_PER_DOMAIN,
        GUARD_GRAD_MIN,
    )
else:
    GUARD_FID=pd.DataFrame()
    GUARD_FID_SUM={
        "phase":"guard_integration",
        "passed":False,
        "sealed":True,
    }

GUARD_FID.to_csv(
    OUT/"guard_custom_autograd_fidelity.csv",
    index=False,
)

print(
    GUARD_FID_SUM
)


## Saved-state census of the custom Function

The census must not observe a full \(B\times T\times H\) FP16/FP32 \(Z\) saved by the custom Function.


In [ ]:
def custom_saved_state_census():
    layer=7
    mode=FROZEN_Z_MODES[layer]

    # Four real sequences for a nontrivial payload.
    chosen=[
        SUPPORT_CPU["language"][0],
        SUPPORT_CPU["language"][1],
        SUPPORT_CPU["code"][0],
        SUPPORT_CPU["code"][1],
    ]

    U_batch=torch.stack(
        [
            native_post_attention_u(
                st,
                layer,
            ).cpu()
            for st in chosen
        ],
        dim=0,
    ).to(
        DEVICE
    )

    params=layer_params(layer)
    old=with_param_grad_flags(params,True)

    records=[]

    def pack_hook(t):
        records.append({
            "shape":list(t.shape),
            "dtype":str(t.dtype),
            "numel":int(t.numel()),
            "bytes":int(
                t.numel()*
                t.element_size()
            ),
        })
        return t

    def unpack_hook(x):
        return x

    try:
        U=(
            U_batch.detach()
            .clone()
            .to(
                model.transformer.h[layer].ln_2.weight.dtype
            )
            .requires_grad_(True)
        )

        with torch.autograd.graph.saved_tensors_hooks(
            pack_hook,
            unpack_hook,
        ):
            out=SHVLN2MLPFunction.apply(
                U,
                *params,
                layer,
                mode,
            )

        loss=out.float().sum()
        loss.backward()

    finally:
        restore_param_grad_flags(params,old)

    full_numel=(
        U_batch.shape[0]*
        SEQ_LEN*
        HIDDEN
    )

    full_float_saved=any(
        row["numel"]==full_numel
        and
        (
            "float16" in row["dtype"]
            or
            "float32" in row["dtype"]
        )
        for row in records
    )

    result={
        "layer":layer,
        "mode":mode,
        "batch":int(U_batch.shape[0]),
        "records":records,
        "full_Z_float_saved":bool(
            full_float_saved
        ),
        "passed":bool(
            not full_float_saved
        ),
    }

    del U_batch
    cleanup()

    return result


CUSTOM_CENSUS=custom_saved_state_census()

(OUT/"custom_autograd_saved_state_census.json").write_text(
    json.dumps(
        CUSTOM_CENSUS,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        CUSTOM_CENSUS,
        indent=2,
    )
)

if not CUSTOM_CENSUS["passed"]:
    raise RuntimeError({
        "custom_saved_state_census_failed":CUSTOM_CENSUS
    })


## Actual CUDA-memory benchmark

The benchmark accumulates 12 real `LN2+MLP` graphs before one backward call so native saved tensors coexist in memory, as they do during training.

Attention is not part of this benchmark; the inputs are real post-Attention states captured from GPT-2.

Reported memory is the **increment above the already-loaded model baseline**.


In [ ]:
@torch.no_grad()
def build_memory_u_batches(batch_size):
    chosen=[]

    half=batch_size//2

    chosen.extend(
        SUPPORT_CPU["language"][:half]
    )

    chosen.extend(
        SUPPORT_CPU["code"][
            :batch_size-half
        ]
    )

    assert len(chosen)==batch_size

    batches={}

    for layer in range(12):
        parts=[]

        for st in chosen:
            parts.append(
                native_post_attention_u(
                    st,
                    layer,
                ).cpu()
            )

        batches[layer]=torch.stack(
            parts,
            dim=0,
        ).contiguous()

        cleanup()

    return batches


MEMORY_U_BATCHES=build_memory_u_batches(
    MEMORY_BENCH_BATCH
)


ALL_MLP_PARAMS=[]

for layer in range(12):
    ALL_MLP_PARAMS.extend(
        layer_params(layer)
    )


def clear_param_grads():
    for p in ALL_MLP_PARAMS:
        p.grad=None


def set_all_param_flags(enabled):
    old=[
        bool(p.requires_grad)
        for p in ALL_MLP_PARAMS
    ]

    for p in ALL_MLP_PARAMS:
        p.requires_grad_(enabled)

    return old


def restore_all_param_flags(flags):
    for p,f in zip(
        ALL_MLP_PARAMS,
        flags,
    ):
        p.requires_grad_(f)


def one_memory_benchmark(kind):
    assert kind in [
        "native",
        "shv",
    ]

    cleanup()
    clear_param_grads()

    flags=set_all_param_flags(
        True
    )

    inputs=[]
    loss=None

    try:
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

        start_alloc=int(
            torch.cuda.memory_allocated()
        )

        start_reserved=int(
            torch.cuda.memory_reserved()
        )

        t0=time.perf_counter()

        for layer in range(12):
            block=model.transformer.h[layer]

            U=(
                MEMORY_U_BATCHES[
                    layer
                ].to(
                    DEVICE,
                    dtype=block.ln_2.weight.dtype,
                )
                .detach()
                .requires_grad_(
                    True
                )
            )

            inputs.append(
                U
            )

            if kind=="native":
                out=(
                    U+
                    block.mlp(
                        block.ln_2(U)
                    )
                )

            else:
                out=SHVLN2MLPFunction.apply(
                    U,
                    *layer_params(layer),
                    layer,
                    FROZEN_Z_MODES[layer],
                )

            # Linear loss: avoid saving output solely for the benchmark loss.
            term=(
                out.float().sum()/
                float(
                    out.numel()
                )
            )

            loss=(
                term
                if loss is None
                else loss+term
            )

        torch.cuda.synchronize()

        forward_ms=(
            time.perf_counter()-
            t0
        )*1000.0

        forward_current=int(
            torch.cuda.memory_allocated()
        )

        forward_peak=int(
            torch.cuda.max_memory_allocated()
        )

        t1=time.perf_counter()

        loss.backward()

        torch.cuda.synchronize()

        backward_ms=(
            time.perf_counter()-
            t1
        )*1000.0

        full_peak=int(
            torch.cuda.max_memory_allocated()
        )

        end_alloc=int(
            torch.cuda.memory_allocated()
        )

        return {
            "kind":kind,
            "start_allocated":start_alloc,
            "start_reserved":start_reserved,
            "forward_resident_delta":int(
                forward_current-
                start_alloc
            ),
            "forward_peak_delta":int(
                forward_peak-
                start_alloc
            ),
            "full_step_peak_delta":int(
                full_peak-
                start_alloc
            ),
            "end_allocated_delta":int(
                end_alloc-
                start_alloc
            ),
            "forward_ms":float(
                forward_ms
            ),
            "backward_ms":float(
                backward_ms
            ),
            "step_ms":float(
                forward_ms+
                backward_ms
            ),
        }

    finally:
        clear_param_grads()
        restore_all_param_flags(
            flags
        )

        del inputs,loss
        cleanup()


# One warmup each.
_ = one_memory_benchmark(
    "native"
)

_ = one_memory_benchmark(
    "shv"
)


BENCH_ROWS=[]

for rep in range(
    MEMORY_BENCH_REPEATS
):
    print(
        "memory benchmark repeat",
        rep,
    )

    BENCH_ROWS.append(
        one_memory_benchmark(
            "native"
        )
    )

    BENCH_ROWS.append(
        one_memory_benchmark(
            "shv"
        )
    )


BENCH=pd.DataFrame(
    BENCH_ROWS
)

BENCH.to_csv(
    OUT/"actual_cuda_memory_benchmark.csv",
    index=False,
)


def median_kind(
    kind,
    col,
):
    return float(
        BENCH[
            BENCH.kind==kind
        ][
            col
        ].median()
    )


native_forward_resident=median_kind(
    "native",
    "forward_resident_delta",
)

shv_forward_resident=median_kind(
    "shv",
    "forward_resident_delta",
)

native_peak=median_kind(
    "native",
    "full_step_peak_delta",
)

shv_peak=median_kind(
    "shv",
    "full_step_peak_delta",
)

native_step=median_kind(
    "native",
    "step_ms",
)

shv_step=median_kind(
    "shv",
    "step_ms",
)


FORWARD_RESIDENT_REDUCTION=(
    1.0-
    shv_forward_resident/
    native_forward_resident
)

FULL_STEP_PEAK_REDUCTION=(
    1.0-
    shv_peak/
    native_peak
)

THROUGHPUT_RATIO=(
    native_step/
    shv_step
)


MEMORY_RESULT={
    "native_forward_resident_delta":native_forward_resident,
    "shv_forward_resident_delta":shv_forward_resident,
    "forward_resident_reduction":FORWARD_RESIDENT_REDUCTION,
    "native_full_step_peak_delta":native_peak,
    "shv_full_step_peak_delta":shv_peak,
    "full_step_peak_reduction":FULL_STEP_PEAK_REDUCTION,
    "native_step_ms":native_step,
    "shv_step_ms":shv_step,
    "throughput_ratio_native_over_shv":THROUGHPUT_RATIO,
    "forward_resident_gate":bool(
        FORWARD_RESIDENT_REDUCTION>=
        FORWARD_RESIDENT_REDUCTION_MIN
    ),
    "full_step_peak_gate":bool(
        FULL_STEP_PEAK_REDUCTION>=
        FULL_STEP_PEAK_REDUCTION_MIN
    ),
}

(OUT/"actual_cuda_memory_result.json").write_text(
    json.dumps(
        MEMORY_RESULT,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        MEMORY_RESULT,
        indent=2,
    )
)


## Formal R5A0 verdict


In [ ]:
GATES={
    "G1_fresh_real_data_contract":bool(
        DATA_CONTRACT[
            "real_sources_loaded"
        ]
        and
        DATA_CONTRACT[
            "no_overlap_with_all_fully_specified_prior_exposed_blocks"
        ]
        and
        DATA_CONTRACT[
            "new_support_promotion_guard_disjoint"
        ]
    ),
    "G2_support_custom_autograd_fidelity":bool(
        SUPPORT_FID_SUM[
            "passed"
        ]
    ),
    "G3_promotion_custom_autograd_fidelity":bool(
        PROMO_FID_SUM.get(
            "passed",
            False,
        )
    ),
    "G4_guard_custom_autograd_fidelity":bool(
        GUARD_FID_SUM.get(
            "passed",
            False,
        )
    ),
    "G5_custom_saved_state_census":bool(
        CUSTOM_CENSUS[
            "passed"
        ]
    ),
    "G6_forward_resident_cuda_memory_reduction":bool(
        MEMORY_RESULT[
            "forward_resident_gate"
        ]
    ),
    "G7_full_step_peak_cuda_memory_reduction":bool(
        MEMORY_RESULT[
            "full_step_peak_gate"
        ]
    ),
}

GATES[
    "R1A_R5A0_QUALIFIES_LIVE_ATTENTION_INTEGRATION"
]=all(
    GATES.values()
)


VERDICT={
    "gates":GATES,
    "frozen_z_modes":FROZEN_Z_MODES,
    "support_fidelity":SUPPORT_FID_SUM,
    "promotion_fidelity":PROMO_FID_SUM,
    "guard_fidelity":GUARD_FID_SUM,
    "custom_saved_state_census":CUSTOM_CENSUS,
    "actual_cuda_memory":MEMORY_RESULT,
    "scope":"MLP/LN2 only; Attention live-autograd replacement not yet integrated",
}

(OUT/"verdict.json").write_text(
    json.dumps(
        VERDICT,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        GATES,
        indent=2,
    )
)


In [ ]:
lines=[
    "# SHV-R1A-R5A0 Result Summary","",
    f"- GPU: {ENV['gpu']}",
    f"- Support fidelity: {SUPPORT_FID_SUM}",
    f"- Promotion fidelity: {PROMO_FID_SUM}",
    f"- Guard fidelity: {GUARD_FID_SUM}",
    f"- Forward-resident memory reduction: {FORWARD_RESIDENT_REDUCTION:.6f}",
    f"- Full-step peak memory reduction: {FULL_STEP_PEAK_REDUCTION:.6f}",
    f"- Native step ms: {native_step:.3f}",
    f"- SHV step ms: {shv_step:.3f}",
    f"- Throughput ratio (native_time / SHV_time): {THROUGHPUT_RATIO:.6f}",
    f"- Qualifies live Attention integration: **{GATES['R1A_R5A0_QUALIFIES_LIVE_ATTENTION_INTEGRATION']}**","",
    "## Gates",
]

for k,v in GATES.items():
    lines.append(
        f"- {k}: {v}"
    )

summary="\n".join(lines)+"\n"

(OUT/"SUMMARY.md").write_text(
    summary,
    encoding="utf-8",
)

print(summary)


In [ ]:
# Package result bundle.

import shutil

RESULT_ZIP=Path(
    "/content/SHV_R1A_R5A0_RESULT_UPLOAD_TO_CHATGPT.zip"
) if Path("/content").exists() else Path(
    "./SHV_R1A_R5A0_RESULT_UPLOAD_TO_CHATGPT.zip"
)

if RESULT_ZIP.exists():
    RESULT_ZIP.unlink()

shutil.make_archive(
    str(
        RESULT_ZIP.with_suffix("")
    ),
    "zip",
    OUT.parent,
    OUT.name,
)

digest=hashlib.sha256(
    RESULT_ZIP.read_bytes()
).hexdigest()

sha_path=RESULT_ZIP.with_suffix(
    RESULT_ZIP.suffix+
    ".sha256"
)

sha_path.write_text(
    f"{digest}  {RESULT_ZIP.name}\n",
    encoding="utf-8",
)

print(
    "RESULT ZIP:",
    RESULT_ZIP
)

print(
    "SHA256:",
    digest
)

try:
    from google.colab import files
    files.download(
        str(
            RESULT_ZIP
        )
    )
    files.download(
        str(
            sha_path
        )
    )
except Exception as e:
    print(
        "Auto-download unavailable:",
        repr(e),
    )


# Decision after R5A0

## PASS

If:

`R1A_R5A0_QUALIFIES_LIVE_ATTENTION_INTEGRATION = True`

then the MLP/LN2 saved state has moved from a representation ruler into a real autograd-residency implementation with measured CUDA-memory benefit.

Proceed to live Attention integration.

Because the qualified Attention state is consumer-specific (`LN1 + probability + V`), it will also require either:
- a custom Attention autograd Function; or
- a fused custom backward path.

A generic tensor-only unpack hook is not sufficient to express that semantic state.

After MLP + Attention live integration, measure full GPT-2 training-step peak VRAM.

## FAIL signatures

### Fidelity passes but forward-resident memory does not fall

The Python packer is retaining unexpected tensors.

Inspect `custom_autograd_saved_state_census.json`.

### Forward-resident drops but full-step peak does not

Encoding or backward recomputation transient workspace dominates peak HBM.

Proceed to chunked/fused pack/decode before Attention integration.

### Memory passes but throughput is poor

This is expected for an unfused Python/PyTorch prototype.

Do not reject the representation.

R5B should move pack/decode and backward kernels to Triton/CUDA.

### Custom-autograd fidelity fails

Do not tune codecs.

The representation has already passed R4R3R4R1.

Fix the live backward implementation until it reproduces the analytical ruler.
